# LAB | First match ≠ right match
## Relevance Scoring & Rerankers for Trustworthy AI + EU AI Act

**Learning goal:** build one retrieval path over two sources and **prove** every answer is grounded in evidence you can inspect.

**Your sources**
- `podcast.txt`: a podcast transcript about the EU *Ethics Guidelines for Trustworthy AI* (2019)
- `eu_ai_act.pdf`: Regulation (EU) 2024/1689, the AI Act (144 pages)

**What you submit:** a `lab_proof.md` containing your code (or a link to this notebook), **one query**, the **retrieved evidence**, the **final answer**, and **one failure or limitation**. There's a template at the end of this notebook.

---

### How this notebook works

| Label | Meaning |
|---|---|
| **PROVIDED** | Code is given. Run it and read it. |
| **YOUR TURN** | Replace the `# TODO` lines with your own code. |
| **OPTIONAL (Advanced)** | Do this if you have time; it's not required to pass. |

Each step ends with a **Done when** check. Don't move on until it passes.

### Prompt discipline
You may use AI to **review, debug or critique** your work, but not to write your final answer or your `lab_proof.md`. A good review prompt:

> *Role: Retrieval reviewer. Task: Review my plan for grounding and verification. Inputs: my sources, query, retrieval path, and evidence. Constraints: Do not produce the final answer. Output: three grounding risks, two trace checks, one simplification.*

---
## Step 0: Environment [PROVIDED]

1. Create a `.env` file next to this notebook containing `OPENAI_API_KEY=...` and `PINECONE_API_KEY=...`
2. Create a folder called `data/` next to this notebook and put `podcast.txt` and `eu_ai_act.pdf` inside it.
3. Run the two cells below.

**Warning:** Never paste API keys into a notebook cell, and never commit your `.env` file.

In [1]:
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q ipywidgets


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


---
## Step 1: Setup & Data Preparation [PROVIDED, except 1d]

**Goal:** turn two very different sources into chunks that each carry **metadata**, meaning labels that say *where the chunk came from*.

**Why metadata matters:** when the system answers *"Fines are up to €35M"*, a reviewer must be able to trace that sentence back to *"EU AI Act, page 115"*. Without metadata you can't trace an answer, and you can't filter results either (Step 5).

Parts 1a–1c are done for you: run each cell and **read the output**. Part 1d (embeddings) is yours to write.

### 1a. Load both sources

In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

C:\Users\sabri\AppData\Local\Temp\ipykernel_25836\320955083.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'data\\eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


### 1b. Chunk with metadata

In [5]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents into chunks and give every chunk a clean, consistent metadata schema."""
    chunks = splitter.split_documents(docs)
    current_section = "recital" if source_type == "regulation" else None
    current_article = None
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,        # "podcast" or "regulation" -> used for filtering in Step 5
            "source_name": source_name,        # human-readable name -> used for citations
            "chunk_id": f"{source_type}-{i}",  # unique id -> lets a reviewer trace an exact chunk
        }
        if source_type == "regulation":
            raw_page = chunk.metadata.get("page")
            page_number = int(raw_page) + 1 if raw_page is not None else None
            if page_number is not None and page_number <= 44:
                current_section = "recital"
                current_article = None
            elif page_number is not None and page_number > 44 and current_section == "recital":
                current_section = "article"

            headings = []
            for match in re.finditer(r"(?im)^[ \t]*Article\s+(\d+)\b", chunk.page_content):
                headings.append((match.start(), "article", int(match.group(1))))
            for match in re.finditer(r"(?im)^[ \t]*ANNEX\s+([IVXLCDM]+)\b", chunk.page_content):
                headings.append((match.start(), "annex", None))
            for _, heading_section, article_number in sorted(headings):
                current_section = heading_section
                current_article = article_number if heading_section == "article" else None

            metadata["section"] = current_section
            if current_section == "article" and current_article is not None:
                metadata["article"] = current_article
        # PDF pages are 0-indexed in the loader; store the page number a human would see
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1
        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


### 1c. Inspect your chunks (always look at your data!)

In [6]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

{'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'section': 'recital', 'page': 21}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of no ...
-----------------------------------------------------------------------------

### 1d. Embeddings model [YOUR TURN]

Set up the embedding model yourself and test it on **one** chunk.

**What to do**
1. Import `OpenAIEmbeddings` from `langchain_openai`.
2. Create a variable called `embeddings` using the model `text-embedding-3-small`. Step 2 uses this exact variable name.
3. Embed the text of `all_chunks[0]` with `embeddings.embed_query(...)` and print the length of the vector.

**Hint:** see **Section 2 of `rag_intro.ipynb`**.
**Note:** you don't embed **all** chunks here. That happens in Step 2, when you upload them to Pinecone, so you only pay for them once.

In [7]:
from langchain_openai import OpenAIEmbeddings

# Show the chunk counts for both sources
print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

# Print a sample from each source with its metadata for traceability
for chunk in (podcast_chunks[0], act_chunks[150]):
    print('Metadata:', chunk.metadata)
    print(chunk.page_content[:300], '...')
    print('-' * 80)

# Embed one chunk and confirm the expected vector dimension
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
sample_embedding = embeddings.embed_query(all_chunks[0].page_content)
embedding_dimension = len(sample_embedding)
print(f"Embedding dimension: {embedding_dimension}")
assert embedding_dimension == 1536, f"Expected 1536 dimensions, got {embedding_dimension}"

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784
Metadata: {'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
Metadata: {'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'section': 'recital', 'page': 21}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of

### Stop and think (write your answers in `lab_proof.md` later)

1. Our metadata has `source_type`, `source_name`, `chunk_id` and `page`. **What's missing?** Think about a user asking *"What does Article 5 say?"* or *"Only show me recitals."* Could you filter on that today?
2. The podcast has no pages. How would a reviewer find where a podcast chunk came from?

You'll come back to this in **Step 5**.

**Done when:** you can see chunk counts for both sources, every chunk you print shows a `metadata` dict, and your test embedding has length **1536**.

---
## Step 2: Embeddings + Baseline Retrieval [YOUR TURN]

**Goal:** build a plain vector search. This is your **baseline**: every later step must beat it, or it isn't worth the extra cost.

**What to do**
1. Create (or connect to) a Pinecone index. Use dimension `1536` and metric `cosine`.
2. Upload `all_chunks` with `PineconeVectorStore.from_documents(...)`. This is the step that embeds every chunk.
3. Write `baseline_search(query, k)`. It should return `(doc, score)` pairs using `similarity_search_with_score`.
4. Run it on the **fixed test queries** below and save the results.

**Hint:** Stuck? Look at **Section 3–4 of `rag_intro.ipynb`** (yesterday's notebook).
**Warning:** Only run the upload cell **once**. Running it again uploads duplicate chunks.

In [8]:
# PROVIDED: fixed test queries. Use the SAME queries in every step so you can compare fairly.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

# PROVIDED: helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [16]:
# YOUR TURN: create / connect to the Pinecone index
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"   # use lowercase letters and dashes only

# Create the index if it does not exist yet; otherwise connect to the existing one
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Created index: {index_name}")
else:
    print(f"Index {index_name} already exists")

# Upload and embed all chunks in Pinecone. Run this cell only once to avoid duplicates.
vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)
print("Documents added to Pinecone vector store")

c:\Users\sabri\miniconda3\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created index: lab-rerankers
Documents added to Pinecone vector store


In [9]:
# Connect to the existing index (no upload, no duplicates)
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)
print(pc.Index(index_name).describe_index_stats())   # should show ~784 vectors


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 784}},
 'total_vector_count': 784,
 'vector_type': 'dense'}


In [10]:
# Baseline vector search using Pinecone similarity scores
def baseline_search(query, k=5):
    return vectorstore.similarity_search_with_score(query, k=k)

baseline_results = {}
for q in TEST_QUERIES:
    baseline_results[q] = baseline_search(q, k=5)
    show_results(q, baseline_results[q], label="Baseline")


=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#4  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#5  score=0.660  [regulation | p.3.0]  'operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in \ncompl'

=== Baseline | Query: What are the seven requirements for trustworthy AI?
#1  score=0.674  [podcast | p

**Done when:** each test query prints 5 results with a score, a source type and a page or chunk id.

**Note it down:** for which query does the **top result** look wrong or only half-relevant? That's your *first match ≠ right match* case.

---
## Step 3: LLM Relevance Scoring [OPTIONAL (Advanced)]

**Goal:** have an LLM judge how relevant each retrieved chunk is, then re-order the results.

**What to do**
1. Retrieve a wider set of candidates (e.g. `k=10`) with `baseline_search`.
2. For each chunk, ask `gpt-4o-mini` to score its relevance to the query from 0 to 1.
3. Combine the two scores, e.g. `combined = 0.3 * similarity + 0.7 * llm_score`, and sort by `combined`.
4. Compare the new top 3 with the baseline top 3.

**Hint:** Reuse `rerank_documents()` from **Section 3 of `rag_pipeline.ipynb`**. It already parses the score and prints it.
**Warning:** This makes 1 LLM call **per chunk, per query**. Count the calls you make.

In [12]:
# YOUR TURN
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def llm_relevance_score(query, doc):
    # TODO: prompt the LLM for a 0-1 score, parse it robustly, return a float
    pass

def llm_rerank(query, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7):
    # TODO: retrieve k_initial -> score each chunk -> combine scores -> sort -> return top k_final as (doc, combined_score)
    pass

**Done when:** you can show at least one query where the order changed, and say whether it got better or worse.

---
## Step 4: Dedicated Reranker (Cross-Encoder or Cohere) [OPTIONAL (Advanced)]

**Goal:** use a model built specifically for reranking. It's usually faster and cheaper than an LLM, and often better.

**Option A: local cross-encoder (no API key needed)**
```
%pip install -q sentence-transformers
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(query, doc.page_content) for doc in docs])
```

**Option B: Cohere Rerank API** (needs `COHERE_API_KEY` in `.env`; see the Cohere Rerank docs)

**What to do:** rerank the same 10 baseline candidates, keep the top 5, and compare with Step 3.

In [ ]:
# YOUR TURN
def crossencoder_rerank(query, k_initial=10, k_final=5):
    # TODO: retrieve k_initial candidates -> score (query, chunk) pairs -> sort -> return top k_final as (doc, score)
    pass

**Done when:** you have a side-by-side of baseline vs reranked top 3 for at least two queries.

---
## Step 5: Metadata Filtering [YOUR TURN]

**Goal:** restrict search to the right part of the knowledge base *before* ranking.

**What to do**
1. Filter by source using Pinecone's filter syntax:
   `vectorstore.similarity_search_with_score(query, k=5, filter={"source_type": {"$eq": "regulation"}})`
2. Run one query twice: once with **only the regulation**, once with **only the podcast**. How do the answers differ?
3. **The real lesson:** try to filter on something your metadata *doesn't* have yet, such as `section` (`"recital"` / `"article"` / `"annex"`) or `article` (`"Article 5"`). You can't.
   → Go back to `chunk_with_metadata` in Step 1, add the field, **delete the index and re-upload** (`pc.delete_index(index_name)`), then try again.

**Hint:** Hints for the EU AI Act: in this PDF the recitals run up to about page 44, the Articles start at *"CHAPTER I"*, and the Annexes start at *"ANNEX I"*. A regex like `r"^Article\s+(\d+)"` on each line can detect article headings.
**Warning:** Pinecone metadata values can't be `None`. Leave the key out instead.

In [11]:
# YOUR TURN: filter by source_type
# TODO
source_filter_query = TEST_QUERIES[1]
regulation_results = vectorstore.similarity_search_with_score(
    source_filter_query, k=5, filter={"source_type": {"$eq": "regulation"}}
)
podcast_results = vectorstore.similarity_search_with_score(
    source_filter_query, k=5, filter={"source_type": {"$eq": "podcast"}}
)
show_results(source_filter_query, regulation_results, label="Regulation only")
show_results(source_filter_query, podcast_results, label="Podcast only")


=== Regulation only | Query: What are the seven requirements for trustworthy AI?
#1  score=0.611  [regulation | p.8.0]  '(27) While the risk-based approach is the basis for a proportionate and effective set of binding rules, it is important '
#2  score=0.587  [regulation | p.22.0]  '(75) Technical robustness is a key requirement for high-risk AI systems. They should be resilient in relation to harmful'
#3  score=0.584  [regulation | p.61.0]  'The requirement for a separate verification by at least two natural persons shall not apply to high-risk AI systems used'
#4  score=0.576  [regulation | p.8.0]  'coherent, trustworthy and human-centric AI, in line with the Charter and with the values on which the Union is \nfounded.'
#5  score=0.554  [regulation | p.61.0]  'benchmarks and measurement methodologies.\n3. The levels of accuracy and the relevant accuracy metrics of high-risk AI sy'

=== Podcast only | Query: What are the seven requirements for trustworthy AI?
#1  score=0.674  [podcas

In [21]:
# YOUR TURN: filter by the NEW metadata field you added (section / article / ...)
# TODO
# First rerun the Step 1b chunking cell so all_chunks contains section/article metadata.
# This one-time rebuild replaces the existing index and uploads the updated chunks.
pc.delete_index(index_name)
pc.create_index(
    name=index_name,
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)
vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name
)

article_query = "What does Article 5 say?"
article_5_results = vectorstore.similarity_search_with_score(
    article_query,
    k=5,
    filter={"source_type": {"$eq": "regulation"}, "article": {"$eq": 5}}
)
show_results(article_query, article_5_results, label="EU AI Act Article 5 only")


=== EU AI Act Article 5 only | Query: What does Article 5 say?
#1  score=0.310  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.251  [regulation | p.52.0]  '(g) the placing on the market, the putting into service for this specific purpose, or the use of biometric categorisatio'
#3  score=0.190  [regulation | p.51.0]  'of a natural person in the areas of workplace and education institutions, except where the use of the AI system is \ninte'
#4  score=0.183  [regulation | p.51.0]  'assessments of natural persons in order to assess or predict the risk of a natural person committing a criminal offence,'
#5  score=0.180  [regulation | p.51.0]  '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation or classification of'


**Done when:** you can run a query that only returns, e.g., *Article* chunks from the EU AI Act, and you can explain why you had to rebuild the index to do it.

---
## Step 6: Complete RAG Pipeline with Reranking [YOUR TURN]

**Goal:** plug your best retrieval method into a full RAG chain that **cites its sources**.

**What to do**
1. Build an LCEL chain: `retrieve → format context → prompt → LLM → answer` (see **Section 5 of `rag_intro.ipynb`**).
2. In `format_context`, put each chunk's source and page **inside** the context, e.g. `[EU AI Act, p.51]`, and tell the LLM to cite them.
3. Make it easy to switch retrieval methods: baseline vs. reranked should be a **1–2 line change**.
4. Make the model say *"I don't know based on the sources"* when the context doesn't contain the answer.

In [13]:
# YOUR TURN
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
LLM_CALLS = {"rerank": 0, "answer": 0}   # simple counter so Step 7 can report the cost

# Reranked retrieval (Steps 3-4 are optional, so a compact LLM reranker lives here):
# retrieve a wider candidate set -> gpt-4o-mini scores each chunk 0-1 -> combine with similarity -> keep top k
rerank_prompt = ChatPromptTemplate.from_template(
    "Rate how relevant the passage is for answering the question.\n"
    "Reply with a single number between 0 and 1 (1 = directly answers it, 0 = unrelated).\n\n"
    "Question: {question}\n\nPassage:\n{passage}\n\nScore:"
)
rerank_chain = rerank_prompt | llm | StrOutputParser()

def llm_score(query, doc):
    LLM_CALLS["rerank"] += 1
    raw = rerank_chain.invoke({"question": query, "passage": doc.page_content})
    match = re.search(r"\d*\.?\d+", raw)
    return min(max(float(match.group()), 0.0), 1.0) if match else 0.0

def reranked_search(query, k=5, k_initial=10, w_sim=0.3, w_llm=0.7):
    candidates = baseline_search(query, k=k_initial)
    scored = [(doc, w_sim * sim + w_llm * llm_score(query, doc)) for doc, sim in candidates]
    return sorted(scored, key=lambda pair: pair[1], reverse=True)[:k]

def citation_label(doc):
    m = doc.metadata
    if "page" in m:   # PDF chunks: page for humans + chunk_id for exact tracing
        return f"{m['source_name']}, p.{int(m['page'])} | {m['chunk_id']}"
    return f"{m['source_name']}, {m['chunk_id']}"

def format_context(results):
    # TODO: join chunks into one string, each prefixed with its source + page/chunk_id
    return "\n\n".join(f"[{citation_label(doc)}]\n{doc.page_content}" for doc, _ in results)

rag_prompt = ChatPromptTemplate.from_template(
    # TODO: write a prompt that answers ONLY from context, cites [source, page], and admits when it doesn't know
    "You answer questions about the EU AI Act and the EU Ethics Guidelines for Trustworthy AI.\n"
    "Use ONLY the context below. Do not use outside knowledge.\n"
    "After every claim, cite the label of the chunk it comes from, copied exactly as it appears "
    "in square brackets in the context, e.g. [EU AI Act (Reg. 2024/1689), p.51 | regulation-310].\n"
    "If the context does not contain the answer, reply exactly: \"I don't know based on the sources.\"\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

rag_chain = rag_prompt | llm | StrOutputParser()

def answer(query, retrieve_fn=baseline_search):   # <- swap retrieve_fn to switch methods
    # TODO: retrieve -> format -> prompt | llm | StrOutputParser() -> return answer AND the results used
    results = retrieve_fn(query)
    LLM_CALLS["answer"] += 1
    response = rag_chain.invoke({"context": format_context(results), "question": query})
    return response, results

# Demo: one query, answer with citations + the exact chunks behind it
demo_query = TEST_QUERIES[3]
demo_answer, demo_results = answer(demo_query, retrieve_fn=reranked_search)   # baseline: retrieve_fn=baseline_search
print(demo_answer)
show_results(demo_query, demo_results, label="Evidence used (reranked)")

Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act (Reg. 2024/1689), p.115 | regulation-665].

=== Evidence used (reranked) | Query: What fines apply for using a prohibited AI practice?
#1  score=0.884  [regulation | p.115.0]  'including start-ups, and their economic viability.\n2. The Member States shall, without delay and at the latest by the da'
#2  score=0.883  [regulation | p.117.0]  'with regard to the same subject matter;\n(e) any similar previous infringements by the Union institution, body, office or'
#3  score=0.875  [regulation | p.117.0]  'undertakings in the protection of their personal data or business secrets.\n6. Funds collected by imposition of fines in '
#4  score=0.872  [regulation | p.108.0]  'establishes that the AI system

**Done when:** one query gives an answer with citations, and you can point to the exact chunk behind each claim.

---
## Step 7: Evaluate: With vs. Without Reranking [YOUR TURN]

**Goal:** decide, with evidence, whether reranking was worth it.

**What to do:** for each test query, read the answers and the retrieved chunks yourself, then fill in the table. You are the judge: no automatic metric.

| Query | Method | Top-1 chunk relevant? (Y/N) | Answer correct? (Y/Partial/N) | Grounded in cited chunk? (Y/N) | Notes |
|---|---|---|---|---|---|
| Q1 | Baseline | | | | |
| Q1 | Reranked | | | | |
| Q2 | Baseline | | | | |
| Q2 | Reranked | | | | |
| … | | | | | |

**Then answer:**
- Did reranking change the answer, or only the order?
- What did it cost (extra calls, seconds per query)? Would a client pay for that?

In [14]:
# YOUR TURN (optional helper): print baseline vs reranked answers side by side, and time each one
import time

comparison = {}
for q in TEST_QUERIES:
    # TODO: time answer(q, baseline_search) and answer(q, <your reranker>), then print both
    comparison[q] = {}
    for label, retrieve_fn in [("Baseline", baseline_search), ("Reranked", reranked_search)]:
        rerank_calls_before = LLM_CALLS["rerank"]
        start = time.perf_counter()
        response, results = answer(q, retrieve_fn)
        seconds = time.perf_counter() - start
        llm_calls = 1 + LLM_CALLS["rerank"] - rerank_calls_before
        comparison[q][label] = {"answer": response, "results": results, "seconds": seconds, "llm_calls": llm_calls}
        show_results(q, results[:3], label=f"{label} top 3 | {seconds:.1f}s | {llm_calls} LLM calls")
        print(f"\n{label} answer:\n{response}\n")
    print("=" * 100)

for label in ["Baseline", "Reranked"]:
    avg_seconds = sum(comparison[q][label]["seconds"] for q in TEST_QUERIES) / len(TEST_QUERIES)
    avg_calls = sum(comparison[q][label]["llm_calls"] for q in TEST_QUERIES) / len(TEST_QUERIES)
    print(f"{label}: {avg_seconds:.1f}s and {avg_calls:.0f} LLM calls per query on average")


=== Baseline top 3 | 2.3s | 1 LLM calls | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'

Baseline answer:
The following AI practices are prohibited under the EU AI Act:

1. The placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciab

### My evaluation (Step 7)

Reranker = `reranked_search`: top 10 baseline candidates, each scored 0-1 by `gpt-4o-mini`, then `0.3 * similarity + 0.7 * llm_score`, keep top 5. I read every answer and its cited chunk myself.

| Query | Method | Top-1 chunk relevant? (Y/N) | Answer correct? (Y/Partial/N) | Grounded in cited chunk? (Y/N) | Notes |
|---|---|---|---|---|---|
| Q1 prohibited practices | Baseline | Y | Partial | Y | Top-1 `regulation-342` starts with Article 4 (AI literacy) but continues into Article 5(1)(a). The answer lists only 1 of the 8 Article 5 practices. |
| Q1 prohibited practices | Reranked | Y | Partial | Y | Same top-1. Ranks 2-3 are now recitals 44 and 29, so the answer lists 3 practices instead of 1. Still incomplete, and 2 of the 3 come from recitals, not the Article. |
| Q2 seven requirements | Baseline | N | N | Y | Top-1 `podcast-2` only says "seven concrete requirements" without listing them. `regulation-50` (recital 27, which lists all seven) is at rank 2, but the model still answered "I don't know". |
| Q2 seven requirements | Reranked | Y | N | Y | The reranker moved `regulation-50` to #1, but the answer is still "I don't know". The Act calls them "principles", not "requirements", so the strict prompt refused. Reranking fixed retrieval but not generation. |
| Q3 HITL vs HOTL | Baseline | Y | Y | Y | `podcast-8` contains both definitions and is the chunk cited. |
| Q3 HITL vs HOTL | Reranked | Y | Y | N | Same answer, but it cites `podcast-7`. That chunk only defines human-in-the-loop. The human-on-the-loop definition is in `podcast-8`. **Wrong citation.** |
| Q4 fines | Baseline | N | Y | Y | Top-1 is the Article 4/5 chunk (no fines). The correct chunk `regulation-665` (Art. 99(3), EUR 35M / 7%) is at rank 3, and the LLM still found it. |
| Q4 fines | Reranked | Y | Y | Y | `regulation-665` moved to #1. Same answer. Neither method mentions the EUR 1.5M fine for Union institutions (Art. 100, `regulation-673`), although it was in the context. |
| Q5 social scoring | Baseline | Y | Y | Y | Cites recital 31 (`regulation-61`). Article 5(1)(c) (`regulation-344`) is also in the context but is not cited. |
| Q5 social scoring | Reranked | Y | Y | Y | Same top 3, same answer. |

**Did reranking change the answer, or only the order?** Mostly only the order. Top-1 improved for Q2 and Q4, and the order changed for every query. The answer changed only for Q1, which got more complete. For Q4, the baseline answer was already right because the LLM reads all 5 chunks, so a better rank-1 didn't matter. For Q2, better retrieval didn't help because the failure was in generation. For Q3, the reranked answer cited the wrong chunk.

**What did it cost?** Baseline: 1 LLM call and about 1.7 s per query. Reranked: 11 LLM calls (10 scores + 1 answer) and about 7.8 s per query, so about 4.5x the latency and 11x the calls. For this corpus with k=5, a client would not pay that for one more complete answer out of five. It would be worth it with a wider candidate pool feeding a smaller final context, or with a cheaper reranker (cross-encoder, or one batched scoring call).

*Numbers come from one run. LLM outputs and timings can vary slightly when re-run.*

---
## Step 8: Write `lab_proof.md` [YOUR TURN]

Create `lab_proof.md` in your repo (this is the **entry point for your reviewer**). Use this template:

```markdown
# Lab Proof: First match ≠ right match

## 1. What I built
One or two sentences, plus a link to this notebook.

## 2. Query
"..."

## 3. Retrieved evidence
| Rank | Source | Page / chunk_id | Score | Snippet |
|---|---|---|---|---|

## 4. Final answer
(paste the answer, including its citations)

## 5. Trace
Which claim in the answer comes from which chunk?

## 6. Baseline vs reranked
What changed? Was it worth the cost?

## 7. Failure or limitation
Where could this system give a plausible but unsupported answer?
```

**Before you submit:**
- [ ] Only files for this lab are in the repo
- [ ] No API keys, `.env` or tokens are committed
- [ ] `lab_proof.md` is the entry point
- [ ] A reviewer can trace **one** answer back to the exact retrieved chunk

---
### Extensions (optional)
- **Hybrid search:** combine BM25 keyword search with vector search, then rerank
- **Cost analysis:** cost per query for baseline vs LLM scoring vs cross-encoder
- **Failure test:** ask something neither source covers. Does your pipeline say "I don't know"?